[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Transactions &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the catalog's models, the twelve books, and the
recording database the savepoint tasks print from. Run it first, then the tasks in order.


In [1]:
import re
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, ForeignKeyField, IntegerField, IntegrityError, Model,
                    SqliteDatabase)

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

class RecordingSqlite(SqliteDatabase):
    """A database that keeps every statement sent through it, with savepoint names masked.

    peewee names each savepoint with a fresh uuid4, so the names differ on every run. They are
    replaced with s... here so that this notebook prints the same thing each time it is run.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.statements = []

    def execute_sql(self, sql, params=None):
        self.statements.append(re.sub(r'"s[0-9a-f]{32}"', '"s..."', " ".join(sql.split())))
        return super().execute_sql(sql, params)


def marks(database):
    """Only the savepoint statements, which is what the nesting looks like from the database."""
    return [line for line in database.statements
            if line.split()[0] in ("SAVEPOINT", "RELEASE", "ROLLBACK")]

db = RecordingSqlite(":memory:")


class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

def build(database):
    """Create the tables and load the catalog, in one transaction."""
    database.create_tables([Author, Book])
    with database.atomic():
        Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
        written = {author.name: author.id for author in Author.select()}
        Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                          for title, author, year, pages in BOOKS]).execute()


build(db)
print("peewee", peewee.__version__, "| the catalog:", Author.select().count(), "authors and",
      Book.select().count(), "books")


peewee 4.5.1 | the catalog: 4 authors and 12 books


**1.** Two writes and a failure, with nothing kept.


In [2]:
marco = Author.get(Author.name == "Marco Pietra")
before = Book.select().count()

try:
    with db.atomic():
        Book.create(title="First Draft", author=marco, year=2024, pages=100)
        Book.create(title="Second Draft", author=marco, year=2024, pages=110)
        raise RuntimeError("the checks after the writes failed")
except RuntimeError:
    pass

print("before:", before, "| after:", Book.select().count())
print("First Draft in the database:", Book.get_or_none(Book.title == "First Draft") is not None)


before: 12 | after: 12
First Draft in the database: False


The exception left the block, so `__exit__` saw it and rolled back. Both writes went away, including
the one that had already succeeded.


**2.** The same pair, with the error caught inside.


In [3]:
with db.atomic():
    Book.create(title="Third Draft", author=marco, year=2024, pages=100)
    Book.create(title="Fourth Draft", author=marco, year=2024, pages=110)
    try:
        raise RuntimeError("the checks after the writes failed")    # caught in here this time
    except RuntimeError as error:
        print("caught:", error)

print("Third Draft in the database: ", Book.get_or_none(Book.title == "Third Draft") is not None)
print("Fourth Draft in the database:", Book.get_or_none(Book.title == "Fourth Draft") is not None)
Book.delete().where(Book.title.endswith("Draft")).execute()


caught: the checks after the writes failed
Third Draft in the database:  True
Fourth Draft in the database: True


2

Catching it inside means nothing reaches `__exit__`, so the block ends the way a block with no
error ends and the row is committed. The difference between this task and the one above it is only
where the `try` was written.


**3.** A decorated function, undone by its second write.


In [4]:
@db.atomic()
def add_author_with_book(name, title):
    """An author and one book, or neither."""
    author = Author.create(name=name, first_book=2024)
    Book.create(title=title, author=author, year=2024, pages=None)  # pages may not be null
    return author


try:
    add_author_with_book("Temporary Name", "A Book")
except IntegrityError as error:
    print("raised:", error)

print("the author in the database:", Author.get_or_none(Author.name == "Temporary Name"))


raised: NOT NULL constraint failed: book.pages
the author in the database: None


The author was written first and still did not survive, because the failure on the next line left
the function, and the function was the transaction.


**4.** A savepoint around each name.


In [5]:
Author.delete().where(Author.name.startswith("Solo")).execute()
db.statements.clear()

kept, skipped = [], []
with db.atomic():
    for number, name in enumerate(["Solo one", "Solo two", "Solo one", "Solo two"]):
        try:
            with db.atomic():
                Author.create(name=name, first_book=2000 + number)
            kept.append(name)
        except IntegrityError:
            skipped.append(name)

print("kept:   ", kept)
print("skipped:", skipped)
print("in the database:", sorted(a.name for a in Author.select().where(Author.name.startswith("Solo"))))
for line in marks(db):
    print("  ", line)


kept:    ['Solo one', 'Solo two']
skipped: ['Solo one', 'Solo two']
in the database: ['Solo one', 'Solo two']
   SAVEPOINT "s...";
   RELEASE SAVEPOINT "s...";
   SAVEPOINT "s...";
   RELEASE SAVEPOINT "s...";
   SAVEPOINT "s...";
   ROLLBACK TO SAVEPOINT "s...";
   SAVEPOINT "s...";
   ROLLBACK TO SAVEPOINT "s...";


Four savepoints, two released and two rolled back. The rollbacks are what make the skipping real:
each bad row is taken back on its own, and the rows around it are untouched.


**5.** Two callables, run after the block.


In [6]:
order = []

with db.atomic():
    Author.create(name="Signal", first_book=1990)
    db.after_commit(lambda: order.append("first registered"))
    db.after_commit(lambda: order.append("second registered"))
    print("inside the block:", order)

print("after the block: ", order)
Author.delete().where(Author.name == "Signal").execute()


inside the block: []
after the block:  ['first registered', 'second registered']


1

Empty inside, both entries outside, and in the order they were registered. The callables are held
until the outermost transaction commits and then run in turn.


**6.** An instance that outlived its row.


In [7]:
ghost = None
try:
    with db.atomic():
        ghost = Author.create(name="Ghost", first_book=1850)
        raise RuntimeError("stop")
except RuntimeError:
    pass

print("the instance says its name is:", repr(ghost.name))
print("the instance says its id is:  ", ghost.id)
print("that row in the database:     ", Author.get_or_none(Author.id == ghost.id))

next_author = Author.create(name="After The Ghost", first_book=1851)
print("the next author written got id:", next_author.id)


the instance says its name is: 'Ghost'
the instance says its id is:   7
that row in the database:      None
the next author written got id: 7


Two of those three lines are wrong about the database, and nothing in Python knows it. The `id` is
the dangerous one, and the last line shows why: the rolled back row released its number, and the
next author written was given the same one. An instance kept past a rollback is not just stale, it
can point at somebody else's row.


---

&#8592; **Back to:** [Transactions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/05-transactions.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
